# 📡 Crypto RSS Sentiment Scraper — BTC & ETH

Scraping **tanpa API key**, **tanpa login** — murni dari RSS feed berita crypto publik.

**Sumber RSS:**
| # | Media | Coverage |
|---|-------|----------|
| 1 | CoinDesk | Berita umum crypto |
| 2 | CoinTelegraph | Market & teknologi |
| 3 | Decrypt | DeFi, NFT, Bitcoin |
| 4 | NewsBTC | Bitcoin-focused |
| 5 | BeInCrypto | Altcoin & market |
| 6 | CryptoSlate | Industri & proyek |
| 7 | Bitcoinist | Bitcoin & macro |
| 8 | AMBCrypto | Altcoin news |
| 9 | CryptoPotato | Market analysis |
| 10 | U.Today | Breaking news |

> ⚡ **Zero setup** — langsung Run All, tidak butuh API key apapun.

---
## 📦 Cell 1 — Install Dependencies

In [ ]:
!pip install -q feedparser vaderSentiment pandas matplotlib seaborn
print('✅ Done!')

---
## ⚙️ Cell 2 — Import & Konfigurasi RSS Feeds

In [ ]:
import feedparser
import requests
import pandas as pd
import time
import re
import warnings
from datetime import datetime, timezone
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')
analyzer = SentimentIntensityAnalyzer()

# ─────────────────────────────────────────────────────────
# DAFTAR RSS FEED
# Tambah/hapus sesuai kebutuhan
# ─────────────────────────────────────────────────────────
RSS_FEEDS = [
    # (Nama Media, URL Feed)
    ('CoinDesk',       'https://www.coindesk.com/arc/outboundfeeds/rss/'),
    ('CoinTelegraph',  'https://cointelegraph.com/rss'),
    ('Decrypt',        'https://decrypt.co/feed'),
    ('NewsBTC',        'https://www.newsbtc.com/feed/'),
    ('BeInCrypto',     'https://beincrypto.com/feed/'),
    ('CryptoSlate',    'https://cryptoslate.com/feed/'),
    ('Bitcoinist',     'https://bitcoinist.com/feed/'),
    ('AMBCrypto',      'https://ambcrypto.com/feed/'),
    ('CryptoPotato',   'https://cryptopotato.com/feed/'),
    ('U.Today',        'https://u.today/rss'),
    ('CryptoNews',     'https://cryptonews.com/news/feed/'),
    ('The Block',      'https://www.theblock.co/rss.xml'),
]

# ─────────────────────────────────────────────────────────
# KEYWORD FILTER per asset
# ─────────────────────────────────────────────────────────
ASSET_KEYWORDS = {
    'BTC': [
        'bitcoin', 'btc', 'satoshi', 'sats',
        'bitcoin etf', 'bitcoin halving', 'bitcoin miner',
    ],
    'ETH': [
        'ethereum', 'eth', 'ether', 'vitalik',
        'ethereum etf', 'gas fee', 'layer 2', 'l2', 'defi',
        'eip-', 'beacon chain', 'proof of stake',
    ],
}

# ─────────────────────────────────────────────────────────
# KONFIGURASI
# ─────────────────────────────────────────────────────────
MAX_ARTICLES_PER_FEED = 50   # max artikel per feed
REQUEST_DELAY        = 0.5   # jeda antar request (detik)
FETCH_TIMEOUT        = 15    # timeout per feed (detik)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/121.0.0.0 Safari/537.36',
    'Accept': 'application/rss+xml, application/xml, text/xml, */*',
    'Accept-Language': 'en-US,en;q=0.9',
}

print('✅ Konfigurasi siap.')
print(f'   Feeds terdaftar : {len(RSS_FEEDS)}')
print(f'   BTC keywords    : {len(ASSET_KEYWORDS["BTC"])}')
print(f'   ETH keywords    : {len(ASSET_KEYWORDS["ETH"])}')


---
## 🧠 Cell 3 — Fungsi Analisis & Parsing

In [ ]:
def clean_html(text: str) -> str:
    """Hapus HTML tag dari teks."""
    if not text:
        return ''
    clean = re.sub(r'<[^>]+>', ' ', text)
    clean = re.sub(r'&[a-z]+;', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean


def get_sentiment(text: str) -> dict:
    """Analisis sentimen VADER — optimal untuk teks berita pendek."""
    if not text or not isinstance(text, str):
        return {'label': 'NEUTRAL', 'compound': 0.0, 'pos': 0.0, 'neu': 1.0, 'neg': 0.0}
    scores = analyzer.polarity_scores(text)
    c = scores['compound']
    label = 'BULLISH' if c >= 0.05 else ('BEARISH' if c <= -0.05 else 'NEUTRAL')
    return {'label': label, 'compound': c,
            'pos': scores['pos'], 'neu': scores['neu'], 'neg': scores['neg']}


def detect_assets(text: str) -> list:
    """Deteksi asset yang disebutkan dalam teks (BTC, ETH, atau keduanya)."""
    text_lower = text.lower()
    found = []
    for asset, keywords in ASSET_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            found.append(asset)
    return found if found else ['GENERAL']


def parse_date(entry) -> str:
    """Parse tanggal dari entry RSS — handle berbagai format."""
    for field in ['published_parsed', 'updated_parsed', 'created_parsed']:
        val = getattr(entry, field, None)
        if val:
            try:
                import calendar
                ts = calendar.timegm(val)
                return datetime.utcfromtimestamp(ts).strftime('%Y-%m-%d %H:%M UTC')
            except:
                pass
    for field in ['published', 'updated']:
        val = getattr(entry, field, None)
        if val:
            return str(val)
    return datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')


def fetch_feed(name: str, url: str) -> list:
    """
    Ambil dan parse satu RSS feed.
    Return: list of article dicts.
    """
    articles = []
    try:
        # feedparser bisa pakai custom handler untuk timeout
        feed = feedparser.parse(
            url,
            agent=HEADERS['User-Agent'],
            request_headers={'Accept': HEADERS['Accept'],
                             'Accept-Language': HEADERS['Accept-Language']},
        )

        status = getattr(feed, 'status', 0)
        entries = feed.entries

        if not entries:
            # Fallback: coba via requests + feedparser.parse(string)
            try:
                r = requests.get(url, headers=HEADERS, timeout=FETCH_TIMEOUT)
                if r.status_code == 200:
                    feed = feedparser.parse(r.content)
                    entries = feed.entries
                    status = r.status_code
            except:
                pass

        if not entries:
            print(f'   ⚠️  {name}: tidak ada artikel (HTTP {status})')
            return []

        for entry in entries[:MAX_ARTICLES_PER_FEED]:
            title   = clean_html(getattr(entry, 'title', '') or '')
            summary = clean_html(getattr(entry, 'summary', '') or
                                 getattr(entry, 'description', '') or '')
            link    = getattr(entry, 'link', '')
            date    = parse_date(entry)
            tags    = [t.get('term','') for t in getattr(entry, 'tags', [])]

            if not title:
                continue

            # Analisis sentimen — pakai title + 100 char pertama summary
            analysis_text = f"{title}. {summary[:100]}" if summary else title
            sentiment = get_sentiment(analysis_text)
            assets    = detect_assets(analysis_text + ' ' + ' '.join(tags))

            articles.append({
                'media':     name,
                'title':     title,
                'summary':   summary[:300] if summary else '',
                'url':       link,
                'date':      date,
                'tags':      ', '.join(tags[:5]),
                'assets':    ', '.join(assets),
                **sentiment,
                'scraped_at': datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC'),
            })

        print(f'   ✅ {name:16s}: {len(articles):3d} artikel')

    except Exception as e:
        print(f'   ❌ {name:16s}: {str(e)[:60]}')

    return articles


# Quick test VADER
tests = [
    'Bitcoin surges to new all-time high as institutions rush in',
    'Ethereum crashes 20% amid regulatory fears and mass selloff',
    'BTC price consolidates near support level',
]
print('🧪 Test VADER:')
for t in tests:
    s = get_sentiment(t)
    e = '📈' if s['label']=='BULLISH' else ('📉' if s['label']=='BEARISH' else '➡️')
    print(f"  {e} [{s['label']:7s}] {s['compound']:+.3f} | {t[:60]}")


---
## 🚀 Cell 4 — Jalankan RSS Scraper

In [ ]:
all_articles = []
failed_feeds = []

print('=' * 55)
print('📡 MULAI SCRAPING RSS FEEDS')
print('=' * 55)

for i, (name, url) in enumerate(RSS_FEEDS, 1):
    print(f'[{i:2d}/{len(RSS_FEEDS)}] {name}...')
    articles = fetch_feed(name, url)
    if articles:
        all_articles.extend(articles)
    else:
        failed_feeds.append(name)
    time.sleep(REQUEST_DELAY)

print()
print('=' * 55)
print(f'✅ SELESAI: {len(all_articles)} artikel dari {len(RSS_FEEDS) - len(failed_feeds)} feed')
if failed_feeds:
    print(f'⚠️  Gagal   : {', '.join(failed_feeds)}')
print('=' * 55)

# Build DataFrame
df_all = pd.DataFrame(all_articles) if all_articles else pd.DataFrame(columns=[
    'media','title','summary','url','date','tags','assets',
    'label','compound','pos','neu','neg','scraped_at'
])

# Filter per asset
df_btc = df_all[df_all['assets'].str.contains('BTC', na=False)].copy()
df_eth = df_all[df_all['assets'].str.contains('ETH', na=False)].copy()

print(f'\n📊 Breakdown:')
print(f'   Total artikel : {len(df_all)}')
print(f'   BTC relevan   : {len(df_btc)}')
print(f'   ETH relevan   : {len(df_eth)}')
print(f'   General/Mixed : {len(df_all) - len(df_btc) - len(df_eth) + len(df_all[df_all["assets"].str.contains("BTC", na=False) & df_all["assets"].str.contains("ETH", na=False)])}')

if not df_all.empty:
    print(f'\n🗓️  Rentang data  : {df_all["date"].min()} → {df_all["date"].max()}')
    print(f'\nPreview 5 artikel pertama:')
    display(df_all[['media','title','assets','label','compound']].head())


---
## 📊 Cell 5 — Analisis Sentimen

In [ ]:
def print_summary(df: pd.DataFrame, asset: str):
    if df.empty:
        print(f'  ⚠️  Tidak ada data untuk {asset}')
        return
    total = len(df)
    bull  = (df['label']=='BULLISH').sum()
    bear  = (df['label']=='BEARISH').sum()
    neut  = total - bull - bear
    avg   = df['compound'].mean()
    sig   = '📈 BULLISH' if avg>=0.05 else ('📉 BEARISH' if avg<=-0.05 else '➡️  NEUTRAL')

    print(f'  ┌─ {asset} ─────────────────────────')
    print(f'  │ Total artikel  : {total}')
    print(f'  │ 🟢 Bullish     : {bull} ({bull/total*100:.1f}%)')
    print(f'  │ 🔴 Bearish     : {bear} ({bear/total*100:.1f}%)')
    print(f'  │ ⚪ Neutral     : {neut} ({neut/total*100:.1f}%)')
    print(f'  │ Avg Compound   : {avg:+.4f}')
    print(f'  └ Signal         : {sig}')
    print()

    # Per-media breakdown
    per_media = df.groupby('media').agg(
        count=('compound','count'),
        avg_compound=('compound','mean')
    ).sort_values('avg_compound', ascending=False)
    print(f'  Per media ({asset}):')
    for media, row in per_media.iterrows():
        sig_m = '📈' if row.avg_compound>=0.05 else ('📉' if row.avg_compound<=-0.05 else '➡️')
        print(f'    {sig_m} {media:16s} {row.avg_compound:+.4f} ({int(row["count"]):3d} artikel)')
    print()

print('\n📋 RINGKASAN SENTIMEN\n')
print_summary(df_btc, 'BTC')
print_summary(df_eth, 'ETH')

# Top articles
for asset, df_asset in [('BTC', df_btc), ('ETH', df_eth)]:
    if df_asset.empty: continue
    print(f'🔝 Top 3 BULLISH — {asset}')
    top = df_asset[df_asset['label']=='BULLISH'].nlargest(3, 'compound')
    for _, r in top.iterrows():
        print(f'  📈 [{r.compound:+.3f}] {r.media} | {r.title[:80]}')
    print(f'\n🔻 Top 3 BEARISH — {asset}')
    bot = df_asset[df_asset['label']=='BEARISH'].nsmallest(3, 'compound')
    for _, r in bot.iterrows():
        print(f'  📉 [{r.compound:+.3f}] {r.media} | {r.title[:80]}')
    print()


---
## 📈 Cell 6 — Visualisasi

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

plt.rcParams.update({
    'figure.facecolor': '#0a0a14',
    'axes.facecolor':   '#0d0d1a',
    'axes.edgecolor':   '#1e1e35',
    'axes.labelcolor':  '#8888aa',
    'xtick.color':      '#555577',
    'ytick.color':      '#555577',
    'text.color':       '#ccccee',
    'grid.color':       '#1a1a2e',
    'grid.linewidth':   0.5,
    'font.family':      'monospace',
})

C = {'BULLISH': '#00e676', 'BEARISH': '#ff1744', 'NEUTRAL': '#37474f'}
A = {'BTC': '#f7931a', 'ETH': '#627eea'}

if df_all.empty:
    print('❌ Tidak ada data untuk divisualisasikan.')
else:
    fig = plt.figure(figsize=(20, 14), facecolor='#0a0a14')
    fig.suptitle('CRYPTO RSS SENTIMENT ANALYSIS — BTC & ETH',
                 fontsize=17, color='#ffffff', fontweight='bold',
                 y=0.98, letterSpacing=1)

    gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

    # ── Row 1: Pie per asset ──
    for idx, (asset, df_a) in enumerate([('BTC', df_btc), ('ETH', df_eth)]):
        ax = fig.add_subplot(gs[0, idx])
        if df_a.empty:
            ax.text(0.5, 0.5, f'No {asset} data', ha='center', va='center',
                    color='#444', transform=ax.transAxes)
            ax.axis('off')
            continue
        counts = df_a['label'].value_counts()
        labels = counts.index.tolist()
        sizes  = counts.values.tolist()
        colors = [C.get(l, '#888') for l in labels]
        wedges, texts, autos = ax.pie(
            sizes, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': '#0a0a14', 'linewidth': 2},
            textprops={'color': '#aaaacc', 'fontsize': 9},
        )
        for a in autos:
            a.set_color('#0a0a14'); a.set_fontweight('bold')
        avg = df_a['compound'].mean()
        sig = 'BULLISH' if avg>=0.05 else ('BEARISH' if avg<=-0.05 else 'NEUTRAL')
        ax.set_title(f'{asset} Distribution\n[{sig}] μ={avg:+.4f}',
                     color=A[asset], fontsize=11, pad=10)

    # ── Row 1 col 2-3: Compound per media ──
    ax_media = fig.add_subplot(gs[0, 2:])
    media_stats = df_all.groupby(['assets','media'])['compound'].mean().reset_index()
    # Simplified: avg compound per media across all articles
    media_avg = df_all.groupby('media')['compound'].mean().sort_values()
    bar_colors = ['#00e676' if v>=0.05 else '#ff1744' if v<=-0.05 else '#37474f'
                  for v in media_avg.values]
    bars = ax_media.barh(media_avg.index, media_avg.values,
                         color=bar_colors, edgecolor='#0a0a14', linewidth=0.5)
    ax_media.axvline(0, color='#333355', linewidth=1)
    ax_media.axvline(0.05, color='#00e67633', linewidth=1, linestyle='--')
    ax_media.axvline(-0.05, color='#ff174433', linewidth=1, linestyle='--')
    ax_media.set_xlabel('Avg Compound Score')
    ax_media.set_title('Avg Sentiment per Media', color='#ffffff', fontsize=11)
    ax_media.grid(axis='x', alpha=0.2)

    # ── Row 2: Histogram compound BTC vs ETH ──
    ax_hist = fig.add_subplot(gs[1, :2])
    for asset, df_a, color in [('BTC',df_btc,'#f7931a'),('ETH',df_eth,'#627eea')]:
        if not df_a.empty:
            ax_hist.hist(df_a['compound'], bins=25, alpha=0.55,
                         color=color, label=asset, edgecolor='#0a0a14')
    ax_hist.axvline(0,    color='#444', lw=1, ls='--')
    ax_hist.axvline(0.05, color='#00e67655', lw=1)
    ax_hist.axvline(-0.05,color='#ff174455', lw=1)
    ax_hist.set_xlabel('Compound Score')
    ax_hist.set_ylabel('Jumlah Artikel')
    ax_hist.set_title('Distribusi Compound Score', color='#ffffff', fontsize=11)
    ax_hist.legend()
    ax_hist.grid(alpha=0.2)

    # ── Row 2: Stacked bar sentiment count per media (top 8) ──
    ax_bar = fig.add_subplot(gs[1, 2:])
    top_media = df_all['media'].value_counts().head(8).index
    df_top = df_all[df_all['media'].isin(top_media)]
    pivot = df_top.groupby(['media','label']).size().unstack(fill_value=0)
    for lbl in ['BULLISH','NEUTRAL','BEARISH']:
        if lbl not in pivot.columns: pivot[lbl] = 0
    pivot[['BULLISH','NEUTRAL','BEARISH']].plot(
        kind='bar', stacked=True, ax=ax_bar,
        color=[C['BULLISH'], C['NEUTRAL'], C['BEARISH']],
        edgecolor='#0a0a14', linewidth=0.3,
    )
    ax_bar.set_xlabel('')
    ax_bar.set_xticklabels(pivot.index, rotation=35, ha='right', fontsize=8)
    ax_bar.set_title('Sentiment Count per Media (Top 8)', color='#ffffff', fontsize=11)
    ax_bar.legend(loc='upper right', fontsize=8, framealpha=0.3)
    ax_bar.grid(axis='y', alpha=0.2)

    # ── Row 3: Summary text cards ──
    for idx, (asset, df_a) in enumerate([('BTC', df_btc), ('ETH', df_eth)]):
        ax = fig.add_subplot(gs[2, idx*2:(idx+1)*2])
        ax.axis('off')
        if df_a.empty:
            ax.text(0.5, 0.5, f'No {asset} data', ha='center', transform=ax.transAxes, color='#444')
            continue
        total = len(df_a)
        bull  = (df_a['label']=='BULLISH').sum()
        bear  = (df_a['label']=='BEARISH').sum()
        neut  = total - bull - bear
        avg   = df_a['compound'].mean()
        sig   = '📈 BULLISH' if avg>=0.05 else ('📉 BEARISH' if avg<=-0.05 else '➡️  NEUTRAL')
        top_src = df_a.groupby('media')['compound'].mean().idxmax()
        txt = (
            f'{asset} SENTIMENT CARD\n\n'
            f'Total Artikel   : {total}\n'
            f'🟢 Bullish       : {bull} ({bull/total*100:.1f}%)\n'
            f'🔴 Bearish       : {bear} ({bear/total*100:.1f}%)\n'
            f'⚪ Neutral       : {neut} ({neut/total*100:.1f}%)\n\n'
            f'Avg Compound    : {avg:+.4f}\n'
            f'SIGNAL          : {sig}\n\n'
            f'Most Bullish Src: {top_src}\n'
            f'Scraped at      : {datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")}'
        )
        ax.text(0.05, 0.95, txt, transform=ax.transAxes, va='top',
                fontsize=10, color='#ccccee', linespacing=1.7,
                bbox=dict(facecolor=A[asset]+'18', edgecolor=A[asset]+'55',
                          boxstyle='round,pad=0.8'))

    plt.savefig('rss_sentiment.png', dpi=150, bbox_inches='tight', facecolor='#0a0a14')
    plt.show()
    print('✅ Chart tersimpan: rss_sentiment.png')


---
## 💾 Cell 7 — Export ke Google Drive

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/crypto_rss_sentiment'
os.makedirs(OUT, exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M')

if not df_all.empty:
    # Semua artikel
    p1 = f'{OUT}/rss_all_{ts}.csv'
    df_all.to_csv(p1, index=False)
    print(f'✅ All articles : {p1}')

    # BTC only
    if not df_btc.empty:
        p2 = f'{OUT}/rss_BTC_{ts}.csv'
        df_btc.to_csv(p2, index=False)
        print(f'✅ BTC articles : {p2}')

    # ETH only
    if not df_eth.empty:
        p3 = f'{OUT}/rss_ETH_{ts}.csv'
        df_eth.to_csv(p3, index=False)
        print(f'✅ ETH articles : {p3}')

    # Chart
    if os.path.exists('rss_sentiment.png'):
        p4 = f'{OUT}/rss_chart_{ts}.png'
        shutil.copy('rss_sentiment.png', p4)
        print(f'✅ Chart        : {p4}')

    # Summary JSON
    summary = {}
    for asset, df_a in [('BTC', df_btc), ('ETH', df_eth)]:
        if df_a.empty: continue
        total = len(df_a)
        avg   = df_a['compound'].mean()
        summary[asset] = {
            'total': total,
            'bullish': int((df_a['label']=='BULLISH').sum()),
            'bearish': int((df_a['label']=='BEARISH').sum()),
            'neutral': int((df_a['label']=='NEUTRAL').sum()),
            'avg_compound': round(float(avg), 4),
            'signal': 'BULLISH' if avg>=0.05 else ('BEARISH' if avg<=-0.05 else 'NEUTRAL'),
            'scraped_at': datetime.utcnow().isoformat(),
        }
    import json
    p5 = f'{OUT}/rss_summary_{ts}.json'
    with open(p5, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f'✅ Summary JSON : {p5}')

    print(f'\n📁 Semua file di: {OUT}')
else:
    print('⚠️  df kosong, tidak ada yang disimpan.')


---
## 🔁 Cell 8 — (Opsional) Auto Refresh Berkala
Scrape ulang secara otomatis tiap N menit, data di-append ke CSV yang sama.

In [ ]:
# ✏️ Ubah sesuai kebutuhan
REFRESH_INTERVAL_MINUTES = 30   # interval refresh
MAX_REFRESH = 4                  # 0 = tidak terbatas

run = 0
log_path = f'{OUT}/rss_live_log.csv'

while MAX_REFRESH == 0 or run < MAX_REFRESH:
    run += 1
    print(f'\n⏰ [{datetime.now().strftime("%H:%M:%S")}] Refresh #{run}')
    new_articles = []
    for name, url in RSS_FEEDS:
        arts = fetch_feed(name, url)
        new_articles.extend(arts)
        time.sleep(REQUEST_DELAY)

    if new_articles:
        df_new = pd.DataFrame(new_articles)
        # Append ke log CSV
        write_header = not os.path.exists(log_path)
        df_new.to_csv(log_path, mode='a', header=write_header, index=False)
        print(f'   +{len(new_articles)} artikel → {log_path}')

        # Update df_btc dan df_eth
        df_btc_new = df_new[df_new['assets'].str.contains('BTC', na=False)]
        df_eth_new = df_new[df_new['assets'].str.contains('ETH', na=False)]
        if not df_btc_new.empty:
            avg = df_btc_new['compound'].mean()
            sig = '📈' if avg>=0.05 else ('📉' if avg<=-0.05 else '➡️')
            print(f'   BTC {sig} {avg:+.4f} ({len(df_btc_new)} artikel)')
        if not df_eth_new.empty:
            avg = df_eth_new['compound'].mean()
            sig = '📈' if avg>=0.05 else ('📉' if avg<=-0.05 else '➡️')
            print(f'   ETH {sig} {avg:+.4f} ({len(df_eth_new)} artikel)')

    if MAX_REFRESH == 0 or run < MAX_REFRESH:
        print(f'   💤 Tunggu {REFRESH_INTERVAL_MINUTES} menit...')
        time.sleep(REFRESH_INTERVAL_MINUTES * 60)

print(f'\n✅ Auto refresh selesai setelah {run} run.')
